# Pathway 3: Speaker Augmentation

This notebook extends a small collection of authentic recordings using data augmentation techniques — without introducing synthetic voices.

## Philosophy

The original version of this repository generated synthetic training data using Google Cloud Text-to-Speech. This created an odd loop: use a corporate voice model to produce data for training a community voice model. The synthetic voices had no connection to the community.

This pathway takes a different approach: **speaker augmentation**. If you have a small set of real, consented recordings, augmentation techniques can expand the dataset by creating variations — while keeping the community's own voices at the center.

**The gold standard remains real recordings.** Te Hiku Media collected 310+ hours from real speakers precisely because augmentation cannot substitute for authentic language data. Use this pathway as a practical measure when real recordings are genuinely limited, not as a shortcut to avoid collecting more.

## What augmentation does and doesn't do

Augmentation creates variations of existing recordings:
- **Speed perturbation**: slightly faster or slower speech
- **Pitch shifting**: slightly higher or lower pitch
- **Additive noise**: simulates different acoustic environments

It does **not** create new words, phrases, or phonemes. The transcript stays the same. Augmented files are always flagged in metadata as derived from a source recording.


## 1. Environment Check

```bash
uv sync --extra dev
```

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")

issues = []
for pkg, mod in [("pydub", "pydub"), ("librosa", "librosa"),
                 ("audiomentations", "audiomentations"), ("soundfile", "soundfile")]:
    try:
        __import__(mod)
    except Exception as e:
        issues.append(f"{pkg}: {e}")
if issues:
    for i in issues: print(i)
    print("\nRun: uv sync --extra dev")
else:
    print("All dependencies available.")

## 2. Configure

In [ ]:
import os

INPUT_DIR = "../test_data/wavs_export_audacity/"  # Source recordings
OUTPUT_DIR = "../test_data/wavs_augmented/"        # Augmented output
SAMPLE_RATE = 22050

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Speed Perturbation

Produces slightly faster and slower versions of each recording. This simulates natural variation in speaking pace and improves model robustness.

In [ ]:
from pydub import AudioSegment
import numpy as np
import soundfile as sf
import librosa
import os

def speed_perturb(wav_path: str, output_dir: str, rates: list[float]) -> list[str]:
    """Generate speed-perturbed versions at each rate."""
    y, sr = librosa.load(wav_path, sr=SAMPLE_RATE)
    file_id = os.path.splitext(os.path.basename(wav_path))[0]
    outputs = []
    for rate in rates:
        y_stretched = librosa.effects.time_stretch(y, rate=rate)
        out_id = f"{file_id}_speed{int(rate*100)}"
        out_path = os.path.join(output_dir, f"{out_id}.wav")
        sf.write(out_path, y_stretched, sr)
        outputs.append(out_id)
    return outputs


SPEED_RATES = [0.9, 1.1]  # 10% slower, 10% faster

wav_files = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith(".wav"))
speed_augmented = []

for fname in wav_files:
    path = os.path.join(INPUT_DIR, fname)
    ids = speed_perturb(path, OUTPUT_DIR, SPEED_RATES)
    source_id = os.path.splitext(fname)[0]
    for aug_id in ids:
        speed_augmented.append({"file_id": aug_id, "source_id": source_id, "technique": "speed_perturbation"})
    print(f"  {fname} → {ids}")

print(f"Generated {len(speed_augmented)} speed-perturbed files.")

## 4. Pitch Shifting

In [ ]:
def pitch_shift(wav_path: str, output_dir: str, semitones: list[float]) -> list[str]:
    """Generate pitch-shifted versions."""
    y, sr = librosa.load(wav_path, sr=SAMPLE_RATE)
    file_id = os.path.splitext(os.path.basename(wav_path))[0]
    outputs = []
    for n_steps in semitones:
        y_shifted = librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)
        sign = "p" if n_steps > 0 else "m"
        out_id = f"{file_id}_pitch{sign}{abs(int(n_steps))}"
        out_path = os.path.join(output_dir, f"{out_id}.wav")
        sf.write(out_path, y_shifted, sr)
        outputs.append(out_id)
    return outputs


PITCH_SEMITONES = [-2, 2]  # ±2 semitones

pitch_augmented = []
for fname in wav_files:
    path = os.path.join(INPUT_DIR, fname)
    ids = pitch_shift(path, OUTPUT_DIR, PITCH_SEMITONES)
    source_id = os.path.splitext(fname)[0]
    for aug_id in ids:
        pitch_augmented.append({"file_id": aug_id, "source_id": source_id, "technique": "pitch_shift"})
    print(f"  {fname} → {ids}")

print(f"Generated {len(pitch_augmented)} pitch-shifted files.")

## 5. Additive Noise

Adds low-level background noise to simulate different recording environments.

In [ ]:
from audiomentations import Compose, AddGaussianNoise, AddGaussianSNR

noise_transform = Compose([
    AddGaussianSNR(min_snr_db=20.0, max_snr_db=40.0, p=1.0),
])

def add_noise(wav_path: str, output_dir: str) -> str:
    y, sr = librosa.load(wav_path, sr=SAMPLE_RATE)
    y_noisy = noise_transform(samples=y, sample_rate=sr)
    file_id = os.path.splitext(os.path.basename(wav_path))[0]
    out_id = f"{file_id}_noise"
    out_path = os.path.join(output_dir, f"{out_id}.wav")
    sf.write(out_path, y_noisy, sr)
    return out_id


noise_augmented = []
for fname in wav_files:
    path = os.path.join(INPUT_DIR, fname)
    aug_id = add_noise(path, OUTPUT_DIR)
    source_id = os.path.splitext(fname)[0]
    noise_augmented.append({"file_id": aug_id, "source_id": source_id, "technique": "additive_noise"})
    print(f"  {fname} → {aug_id}")

print(f"Generated {len(noise_augmented)} noise-augmented files.")

## 6. Update Metadata

Augmented files must be flagged in metadata. The `provenance_note` field records the source file ID so no downstream user mistakes an augmented recording for original speech.

In [ ]:
import pandas as pd

METADATA_PATH = "../metadata/metadata_template.csv"

# Load existing metadata
meta = pd.read_csv(METADATA_PATH, dtype=str)

# Build rows for augmented files, inheriting consent/protocol from source
all_augmented = speed_augmented + pitch_augmented + noise_augmented
new_rows = []

for aug in all_augmented:
    source_row = meta[meta["file_id"] == aug["source_id"]]
    new_row = {
        "file_id": aug["file_id"],
        "transcript": source_row["transcript"].values[0] if len(source_row) > 0 else "",
        "speaker_id": source_row["speaker_id"].values[0] if len(source_row) > 0 else "",
        "language": source_row["language"].values[0] if len(source_row) > 0 else "",
        "consent_tier": source_row["consent_tier"].values[0] if len(source_row) > 0 else "",
        "cultural_protocol": source_row["cultural_protocol"].values[0] if len(source_row) > 0 else "",
        "knowledge_keeper_reviewed": source_row["knowledge_keeper_reviewed"].values[0] if len(source_row) > 0 else "false",
        "exclude_from_training": "false",
        "exclude_reason": "",
        "recorded_by": source_row["recorded_by"].values[0] if len(source_row) > 0 else "",
        "recording_date": source_row["recording_date"].values[0] if len(source_row) > 0 else "",
        "provenance_note": f"augmented from {aug['source_id']} ({aug['technique']})",
    }
    new_rows.append(new_row)

aug_df = pd.DataFrame(new_rows)
combined = pd.concat([meta, aug_df], ignore_index=True)
combined.to_csv(METADATA_PATH, index=False, encoding="utf-8")

print(f"Added {len(new_rows)} augmented rows to {METADATA_PATH}")
print(f"Total rows: {len(combined)}")

## Next Steps

- Export the final dataset: [05_export_ljspeech.ipynb](05_export_ljspeech.ipynb)

**Note:** Review augmented files in [03_snr_quality_analysis.ipynb](03_snr_quality_analysis.ipynb) before exporting — noise augmentation can push SNR below acceptable thresholds.